### Notebook layout
- Imports & data
- Aggregated validation results for all the CNN variants
- Aggregated validation results for all the Claude variants
- Point estimates and confidence interval for move recognition accuracy for best CNN variant and best Claude variant on test set
- Hypothesis test: is best CNN variant better than best Claude variant?

### Imports & data

In [1]:
import polars as pl
pl.Config.set_tbl_rows(30)

polars.config.Config

In [2]:
df = pl.read_csv("results.csv")

### Aggregated validation results for all the CNN variants

In [3]:
(
    df.filter(pl.col("split").eq("val"), pl.col("method").eq("cnn"))
    .group_by("model_version")
    .agg(
        pl.col("correct_square_mean").mean(),
        pl.col("correct_board_mean").mean(),
        pl.len().alias("n")
    )
    .sort("correct_board_mean", descending=True)
)

model_version,correct_square_mean,correct_board_mean,n
str,f64,f64,u32
"""optimised_plus_prior_correctio…",0.739553,0.947831,5
"""optimised""",0.727543,0.910303,5
"""optimised_10k""",0.708313,0.909649,5
"""square_global""",0.716104,0.859952,5
"""square_per_square""",0.759768,0.858538,5
"""optimised_5k""",0.63859,0.788188,5
"""none_global""",0.602451,0.307201,5


### Aggregated validation results for all the Claude variants

In [4]:
(
    df.filter(pl.col("split").eq("val"), pl.col("method").ne("cnn"))
    .group_by("method", "model_version", "prompt_version", "reasoning", "effort")
    .agg(
        pl.col("correct_square_mean").mean(),
        pl.col("correct_board_mean").mean(),
        pl.len().alias("n")
    )
    .sort("correct_board_mean", descending=True)
)

method,model_version,prompt_version,reasoning,effort,correct_square_mean,correct_board_mean,n
str,str,i64,str,str,f64,f64,u32
"""square_logits""","""claude-opus-5""",1,"""thinking""","""low""",0.50854,0.326845,5
"""square_logits""","""claude-opus-5""",1,"""none""","""high""",0.524569,0.322662,5
"""square_logits""","""claude-opus-5""",1,"""thinking""","""medium""",0.448652,0.294676,5
"""square_logits""","""claude-opus-5""",1,"""thinking""","""high""",0.425266,0.27937,5
"""square_label""","""claude-opus-5""",1,"""thinking""","""medium""",0.411346,0.197813,5
"""square_label""","""claude-opus-5""",1,"""none""","""high""",0.550072,0.17716,5
"""square_label""","""claude-opus-5""",1,"""thinking""","""high""",0.408221,0.16852,5
"""square_label""","""claude-sonnet-5""",1,"""thinking""","""high""",0.42104,0.129031,5
"""square_label""","""claude-sonnet-5""",1,"""none""","""high""",0.42352,0.128378,5


### Point estimates and confidence interval for move recognition accuracy for best CNN variant and best Claude variant on test set

In [5]:
# Point estimates
cnn_rows = df.filter(pl.col("model_version").eq("optimised_plus_prior_correction"), pl.col("split").eq("test"))
claude_rows = df.filter(pl.col("model_version").eq("claude-opus-5"), pl.col("method").eq("square_logits"), pl.col("effort").eq("low"), pl.col("split").eq("test"))
print(f"Average CNN accuracy across {cnn_rows.height} test games: {cnn_rows["correct_board_mean"].mean():.1%}")
print(f"Average Claude accuracy across {claude_rows.height} test games: {claude_rows["correct_board_mean"].mean():.1%}")

Average CNN accuracy across 11 test games: 94.6%
Average Claude accuracy across 11 test games: 22.7%


In [6]:
import numpy as np
def bootstrap_ci(values: np.ndarray, B=10000, alpha=0.05, method="percentile"):
    point_estimate = values.mean()
    rng = np.random.default_rng(seed=123)
    samples = rng.choice(values, size=(B, values.size))
    estimates = samples.mean(axis=1)
    lower_q, upper_q = np.quantile(estimates, q=[alpha/2, 1 - alpha/2])

    if method != "percentile": # alternative to percentile: standard bootstrap
        return f"{2 * point_estimate - upper_q:1%} to {2 * point_estimate - lower_q:.1%}"
    return f"{lower_q:.1%} to {upper_q:.1%}"

print("95% CI for CNN accuracy: " + bootstrap_ci(cnn_rows["correct_board_mean"].to_numpy()))
print("95% CI for Claude accuracy: " + bootstrap_ci(claude_rows["correct_board_mean"].to_numpy()))

95% CI for CNN accuracy: 88.3% to 100.0%
95% CI for Claude accuracy: 13.6% to 31.9%


### Hypothesis test: is best CNN variant better than best Claude variant?
- Definition of "CNN variant and Claude variant are equally good": the probability that on a new game, or new "setup", CNN obtains a higher accuracy than Claude is 0.5. This is H0.
- The test statistic is the proportion of games for which CNN accuracy is higher.
- Under H0, the test statistic is known: it has the same distribution as the number of heads generated from 11 flips of a fair coin. 

In [7]:
import scipy.stats as stats

n_cnn_better = 0
results = {}

for setup in cnn_rows["setup_id"].to_list():
    results[setup] = {
        "cnn_acc": cnn_rows.filter(pl.col("setup_id").eq(setup))["correct_board_mean"].item(),
        "claude_acc": claude_rows.filter(pl.col("setup_id").eq(setup))["correct_board_mean"].item()
    }
    results[setup]["d"] = results[setup]["cnn_acc"] - results[setup]["claude_acc"]
    if results[setup]["d"] > 0:
        n_cnn_better += 1

test_stat = n_cnn_better / len(results)
n = len(results)
as_extreme = []

for i in range(n + 1):
    if np.abs(i / n - 0.5) >= np.abs(test_stat - 0.5):
        as_extreme.append(i) 

p_val = stats.Binomial(n=n, p=0.5).pmf(as_extreme).sum()

print(f"p-value: {p_val:.4f}")

p-value: 0.0010
